# CLM v2 Validation 001b — Closed-Loop Scaffold Handoff

Runs the preregistered trajectory-aware scaffold homotopy without changing the CLM v2 architecture. Direct alpha=0 replacement after local imitation is telemetry only; the registered handoff is alpha 0.75 → 0.50 → 0.25 → 0.00.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'research/clm-v2-validation-001b')
os.chdir('/kaggle/working')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth','1','--branch',REF,'https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
subprocess.run(['git','log','-1','--oneline'], cwd=ROOT, check=True)


In [ ]:
import torch
print({'python':sys.version.split()[0], 'torch':torch.__version__, 'cuda':torch.version.cuda, 'gpu_count':torch.cuda.device_count(), 'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'CUDA is required'
subprocess.run([sys.executable,'-m','pytest','tests/research/01-foundations/test_overcomplete_cellular_textnca.py','tests/research/03-routing-and-growth/test_clm_v2_validation.py','tests/research/03-routing-and-growth/test_clm_v2_validation_001b.py','-q'], cwd=ROOT, check=True)


In [ ]:
# Default behavior resumes completed/partial replicates. Use --fresh only for an intentional clean rerun.
subprocess.run([sys.executable,'scripts/research/run_clm_v2_validation_001b.py'], cwd=ROOT, check=True)


In [ ]:
import json, pandas as pd
from IPython.display import Image, display
OUT = ROOT/'results'/'clm-v2-validation-001b-closed-loop-handoff'
print(json.dumps(json.loads((OUT/'decision.json').read_text()), indent=2))
for name in ['progression.csv','arms.csv','router-diagnostics.csv']:
    path = OUT/name
    if path.exists() and path.stat().st_size > 0:
        frame = pd.read_csv(path)
        if frame.empty:
            print(f'{name}: no rows (experiment stopped before this stage)')
        else:
            display(frame)
    else:
        print(f'{name}: unavailable')


In [ ]:
for name in ['scaffold-handoff.png','local-imitation.png','handoff-recovery.png','quality-vs-k.png','routing-controls.png','routing-variation.png','program-usage.png','program-coactivation.png','capacity-vs-compute.png']:
    path = OUT/name
    if path.exists(): display(Image(filename=str(path)))


In [ ]:
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable,'scripts/research/publish_clm_v2_validation_001b_results.py','--push'], cwd=ROOT, check=True)
else:
    print('PUBLISH=False. Review decision/tables/plots first, then rerun this cell with PUBLISH=True.')
